# 🤖 Machine Learning Demo Apps
## Time Series Forecasting · Customer Segmentation · CNN Image Classification

Notebook ini berisi tiga aplikasi Streamlit interaktif untuk kelas Machine Learning.

📖 **Dokumentasi lengkap:** [github.com/afrizalmeka/ml-demo-apps](https://github.com/afrizalmeka/ml-demo-apps)

| App | Deskripsi | Port |
|-----|-----------|------|
| 🔮 **Time Series Forecasting** | Peramalan deret waktu dengan Supervised Learning | 8501 |
| 👥 **Customer Segmentation** | Segmentasi pelanggan dengan KMeans Clustering | 8502 |
| 🧠 **CNN Image Classification** | Klasifikasi gambar dengan Deep Learning (PyTorch) — Sesi 4 | 8503 |

### Langkah Penggunaan:
1. Jalankan **Cell 1** — install dependencies
2. Jalankan **Cell 2** — masukkan ngrok authtoken Anda
3. Jalankan **Cell 3** — tulis file `app_timeseries.py`
4. Jalankan **Cell 4** — tulis file `app_segmentasi.py`
5. Jalankan **Cell 5** — tulis file `app_cnn.py`
6. Jalankan **Cell 6** — untuk membuka App 1 (Time Series)
7. Jalankan **Cell 7** — untuk membuka App 2 (Segmentasi)
8. Jalankan **Cell 8** — untuk membuka App 3 (CNN - Sesi 4)
9. *(Opsional)* Jalankan **Cell 9** — untuk menghentikan semua tunnel

---
## ⚙️ CELL 1 — Install Dependencies

In [ ]:
!pip install streamlit pyngrok plotly scikit-learn pandas numpy yfinance torch torchvision Pillow

---
## 🔑 CELL 2 — Setup Ngrok Authtoken

> **Dapatkan token gratis di:** https://dashboard.ngrok.com/get-started/your-authtoken  
> Ganti `YOUR_NGROK_AUTHTOKEN_HERE` dengan token Anda.

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN_HERE")

---
## 🔮 CELL 3 — Tulis File `app_timeseries.py`

In [ ]:
%%writefile app_timeseries.py
# ============================================================
# APP 1: TIME SERIES FORECASTING DEMO
# Supervised Learning untuk peramalan deret waktu
# ============================================================

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import yfinance as yf
import io
import warnings
warnings.filterwarnings('ignore')

# ── Konfigurasi halaman ──────────────────────────────────────
st.set_page_config(
    page_title="🔮 Time Series Forecasting Demo",
    page_icon="🔮",
    layout="wide"
)

st.title("🔮 Time Series Forecasting")
st.markdown(
    "Demo interaktif peramalan deret waktu menggunakan "
    "Supervised Learning. Ubah parameter di sidebar untuk "
    "melihat bagaimana model bereaksi."
)

# ── Fungsi: buat data demo sintetis ─────────────────────────
def buat_data_demo():
    """Generate 36 bulan data penjualan sintetis yang reproducible."""
    rng = np.random.RandomState(42)
    n = 36
    tanggal = pd.date_range(start="2022-01", periods=n, freq="MS")
    tren = np.linspace(100, 250, n)
    # Seasonal: naik bulan 11-12, turun bulan 1-2
    seasonal = np.array([
        -20 if m in [1, 2] else (40 if m in [11, 12] else 0)
        for m in tanggal.month
    ])
    noise = rng.normal(0, 10, n)
    nilai = tren + seasonal + noise
    # Satu anomali kecil di sekitar bulan ke-18
    nilai[17] += 60
    df = pd.DataFrame({"tanggal": tanggal, "nilai": np.round(nilai, 2)})
    return df

# ── Fungsi: buat lag features ────────────────────────────────
def buat_lag_features(df, kolom_nilai, n_lag):
    """Tambahkan kolom lag_1 sampai lag_n ke dataframe."""
    df_lag = df.copy()
    for i in range(1, n_lag + 1):
        df_lag[f"lag_{i}"] = df_lag[kolom_nilai].shift(i)
    return df_lag.dropna().reset_index(drop=True)

# ── Fungsi: pilih model ──────────────────────────────────────
def pilih_model(nama_model):
    if nama_model == "Linear Regression":
        return LinearRegression()
    elif nama_model == "Ridge Regression":
        return Ridge(alpha=1.0)
    else:
        return DecisionTreeRegressor(random_state=42)

# ── Fungsi: hitung metrik ────────────────────────────────────
def hitung_metrik(y_aktual, y_prediksi):
    mae = mean_absolute_error(y_aktual, y_prediksi)
    rmse = np.sqrt(mean_squared_error(y_aktual, y_prediksi))
    r2 = r2_score(y_aktual, y_prediksi)
    return mae, rmse, r2

# ── Fungsi: hitung baseline naive ───────────────────────────
def hitung_baseline_naive(y_aktual, y_train_last):
    """Baseline: prediksi = nilai sebelumnya (naive forecast)."""
    y_naive = np.concatenate([[y_train_last], y_aktual[:-1]])
    mae = mean_absolute_error(y_aktual, y_naive)
    rmse = np.sqrt(mean_squared_error(y_aktual, y_naive))
    r2 = r2_score(y_aktual, y_naive)
    return mae, rmse, r2

# ── Sidebar ──────────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Pengaturan")

    # Pilihan sumber data
    sumber_data = st.radio(
        "Pilih Sumber Data",
        options=[
            "📊 Data Demo (Penjualan Sintetis)",
            "📈 Data Saham Live (via yfinance)",
            "📁 Upload CSV Sendiri"
        ]
    )

    df_data = None
    nama_dataset = "Data Demo"
    interval_data = "bulanan"

    # ── Opsi 1: Data Demo ────────────────────────────────────
    if sumber_data == "📊 Data Demo (Penjualan Sintetis)":
        df_data = buat_data_demo()
        nama_dataset = "Penjualan Sintetis"
        interval_data = "bulanan"

    # ── Opsi 2: Data Saham ───────────────────────────────────
    elif sumber_data == "📈 Data Saham Live (via yfinance)":
        ticker_input = st.text_input(
            "Kode Saham",
            value="BBCA.JK",
            placeholder="Contoh: BBCA.JK, TLKM.JK, AAPL, TSLA"
        )
        periode_map = {"1 Tahun": "1y", "2 Tahun": "2y", "3 Tahun": "3y"}
        periode_label = st.selectbox("Periode Data", list(periode_map.keys()))

        interval_map = {"Harian": "1d", "Mingguan": "1wk", "Bulanan": "1mo"}
        interval_label = st.selectbox("Interval", list(interval_map.keys()))

        if st.button("🔍 Ambil Data"):
            try:
                raw = yf.download(
                    ticker_input.strip(),
                    period=periode_map[periode_label],
                    interval=interval_map[interval_label],
                    progress=False
                )
                if raw.empty or "Close" not in raw.columns:
                    raise ValueError(f"Ticker '{ticker_input}' tidak ditemukan atau tidak ada data.")
                raw = raw.reset_index()
                kolom_tanggal = "Datetime" if "Datetime" in raw.columns else "Date"
                df_saham = raw[[kolom_tanggal, "Close"]].copy()
                # Flatten MultiIndex kolom jika ada
                df_saham.columns = ["tanggal", "nilai"]
                df_saham["nilai"] = pd.to_numeric(df_saham["nilai"], errors="coerce")
                df_saham = df_saham.dropna().reset_index(drop=True)
                st.session_state["df_saham"] = df_saham
                st.session_state["nama_saham"] = ticker_input.strip()
                st.session_state["interval_saham"] = interval_label
                st.info(f"✅ Berhasil mengambil {len(df_saham)} baris data untuk **{ticker_input.strip()}**")
            except Exception as e:
                st.error(f"❌ Gagal mengambil data: {e}\n\nMenggunakan data demo sebagai pengganti.")
                st.session_state["df_saham"] = buat_data_demo()
                st.session_state["nama_saham"] = "Demo (fallback)"
                st.session_state["interval_saham"] = "Bulanan"

        if "df_saham" in st.session_state:
            df_data = st.session_state["df_saham"]
            nama_dataset = f"Saham {st.session_state.get('nama_saham', ticker_input)}"
            interval_data = st.session_state.get("interval_saham", "Bulanan").lower()
        else:
            df_data = buat_data_demo()
            nama_dataset = "Data Demo (belum fetch)"
            interval_data = "bulanan"

        st.warning(
            "⚠️ Data saham bersifat sangat fluktuatif (noisy). Model sederhana "
            "seperti Linear Regression mungkin menghasilkan akurasi rendah — "
            "hal ini wajar dan justru menjadi pembelajaran penting: tidak semua "
            "model cocok untuk semua jenis data."
        )

    # ── Opsi 3: Upload CSV ───────────────────────────────────
    else:
        st.markdown(
            "**Format CSV:** 2 kolom wajib — kolom tanggal "
            "(format `YYYY-MM` atau `YYYY-MM-DD`) dan kolom nilai numerik."
        )
        file_upload = st.file_uploader("Upload file CSV", type=["csv"])
        if file_upload is not None:
            try:
                df_upload = pd.read_csv(file_upload)
                st.write("**Preview 5 baris pertama:**")
                st.dataframe(df_upload.head())
                kolom_tersedia = list(df_upload.columns)
                kolom_tgl = st.selectbox("Pilih kolom tanggal", kolom_tersedia, index=0)
                kolom_val = st.selectbox(
                    "Pilih kolom nilai",
                    kolom_tersedia,
                    index=min(1, len(kolom_tersedia) - 1)
                )
                try:
                    df_csv = df_upload[[kolom_tgl, kolom_val]].copy()
                    df_csv.columns = ["tanggal", "nilai"]
                    df_csv["tanggal"] = pd.to_datetime(df_csv["tanggal"])
                    df_csv["nilai"] = pd.to_numeric(df_csv["nilai"], errors="coerce")
                    df_csv = df_csv.dropna().sort_values("tanggal").reset_index(drop=True)
                    if len(df_csv) < 10:
                        raise ValueError("Data terlalu sedikit (minimal 10 baris).")
                    df_data = df_csv
                    nama_dataset = file_upload.name
                    interval_data = "bulanan"
                except Exception as e:
                    st.error(f"❌ Gagal memproses CSV: {e}\n\nMenggunakan data demo.")
                    df_data = buat_data_demo()
                    nama_dataset = "Data Demo (fallback)"
                    interval_data = "bulanan"
            except Exception as e:
                st.error(f"❌ Gagal membaca file: {e}")
                df_data = buat_data_demo()
                nama_dataset = "Data Demo (fallback)"
                interval_data = "bulanan"
        else:
            df_data = buat_data_demo()
            nama_dataset = "Data Demo (menunggu upload)"
            interval_data = "bulanan"

    st.divider()

    # Pilihan model
    model_dipilih = st.selectbox(
        "Pilih Model",
        ["Linear Regression", "Ridge Regression", "Decision Tree Regressor"]
    )

    # Jumlah periode forecast
    n_forecast = st.slider("Jumlah Bulan/Periode Ramalan ke Depan", 1, 12, 6)

    # Jumlah lag features
    n_lag = st.slider("Jumlah Lag Features", 1, 12, 3)

    # Penjelasan lag features
    with st.expander("ℹ️ Apa itu Lag Features?"):
        st.markdown("""
**Lag features** adalah nilai dari waktu sebelumnya yang digunakan sebagai fitur input model.

**Contoh dengan lag=3:**
- Fitur: `nilai_t-1`, `nilai_t-2`, `nilai_t-3`
- Target: `nilai_t`

Semakin banyak lag → model "mengingat" lebih jauh ke masa lalu.  
Terlalu sedikit lag → model kurang informasi.  
Terlalu banyak lag → model bisa overfit pada data kecil.

*(Lag features = cara mengubah time series menjadi supervised learning problem.)*
        """)

# ── Pastikan df_data tidak None ──────────────────────────────
if df_data is None or len(df_data) == 0:
    st.error("Data tidak tersedia. Menggunakan data demo.")
    df_data = buat_data_demo()
    nama_dataset = "Data Demo (fallback)"
    interval_data = "bulanan"

st.info(f"📂 Sumber data aktif: **{nama_dataset}** — {len(df_data)} baris data")

# ── Proses: buat lag features & split data ───────────────────
df_lag = buat_lag_features(df_data, "nilai", n_lag)
fitur_cols = [f"lag_{i}" for i in range(1, n_lag + 1)]

# Pastikan cukup data setelah pembuatan lag
if len(df_lag) < 5:
    st.error(
        f"❌ Data terlalu sedikit setelah pembuatan {n_lag} lag features. "
        "Kurangi jumlah lag atau gunakan data yang lebih panjang."
    )
    st.stop()

# Train/test split 80/20 (time-aware, no shuffle)
split_idx = int(len(df_lag) * 0.8)
if split_idx < 2:
    split_idx = 2

df_train = df_lag.iloc[:split_idx]
df_test = df_lag.iloc[split_idx:]

X_train = df_train[fitur_cols].values
y_train = df_train["nilai"].values
X_test = df_test[fitur_cols].values
y_test = df_test["nilai"].values

# Training model
model = pilih_model(model_dipilih)
model.fit(X_train, y_train)
y_pred_test = model.predict(X_test)

# ── Forecast iteratif ke depan ───────────────────────────────
# Buffer: n_lag nilai terakhir dari seluruh data historis
buffer = list(df_data["nilai"].values[-n_lag:])
forecast_values = []

for _ in range(n_forecast):
    fitur_next = np.array(buffer[-n_lag:][::-1]).reshape(1, -1)
    pred = model.predict(fitur_next)[0]
    forecast_values.append(pred)
    buffer.append(pred)

# Buat label tanggal forecast (lanjut dari tanggal terakhir)
tgl_terakhir = pd.Timestamp(df_data["tanggal"].values[-1])

def buat_tanggal_forecast(tgl_terakhir, n, interval):
    """Generate n tanggal berikutnya sesuai interval data."""
    hasil = []
    tgl = tgl_terakhir
    for _ in range(n):
        if "harian" in interval:
            tgl = tgl + pd.tseries.offsets.BDay(1)
        elif "mingguan" in interval:
            tgl = tgl + pd.DateOffset(weeks=1)
        else:
            tgl = tgl + pd.DateOffset(months=1)
        hasil.append(tgl)
    return hasil

tanggal_forecast = buat_tanggal_forecast(tgl_terakhir, n_forecast, interval_data)

df_forecast = pd.DataFrame({
    "tanggal": tanggal_forecast,
    "nilai_prediksi": np.round(forecast_values, 2)
})

# ── TABS ─────────────────────────────────────────────────────
tab1, tab2, tab3 = st.tabs(["📈 Data & Prediksi", "📊 Evaluasi Model", "🔍 Data Mentah"])

# ── Tab 1: Data & Prediksi ────────────────────────────────────
with tab1:
    # Siapkan data untuk chart
    tgl_train = df_train["tanggal"].values
    tgl_test = df_test["tanggal"].values

    fig = go.Figure()

    # Garis biru solid: data train
    fig.add_trace(go.Scatter(
        x=tgl_train,
        y=df_train["nilai"].values,
        mode="lines",
        name="Train (Historis)",
        line=dict(color="royalblue", width=2),
        hovertemplate="%{x|%Y-%m-%d}<br>Nilai: %{y:.2f}<extra>Train</extra>"
    ))

    # Garis hijau solid: data aktual test
    fig.add_trace(go.Scatter(
        x=tgl_test,
        y=y_test,
        mode="lines",
        name="Test (Aktual)",
        line=dict(color="seagreen", width=2),
        hovertemplate="%{x|%Y-%m-%d}<br>Nilai: %{y:.2f}<extra>Test Aktual</extra>"
    ))

    # Garis merah putus-putus: prediksi test
    fig.add_trace(go.Scatter(
        x=tgl_test,
        y=y_pred_test,
        mode="lines",
        name="Prediksi (Test)",
        line=dict(color="crimson", width=2, dash="dash"),
        hovertemplate="%{x|%Y-%m-%d}<br>Prediksi: %{y:.2f}<extra>Prediksi Test</extra>"
    ))

    # Garis oranye putus-putus: forecast ke depan
    fig.add_trace(go.Scatter(
        x=df_forecast["tanggal"],
        y=df_forecast["nilai_prediksi"],
        mode="lines+markers",
        name="Forecast",
        line=dict(color="darkorange", width=2, dash="dot"),
        marker=dict(size=6),
        hovertemplate="%{x|%Y-%m-%d}<br>Forecast: %{y:.2f}<extra>Forecast</extra>"
    ))

    # Shaded area: batas train/test
    if len(tgl_test) > 0:
        x_split_train_test = str(pd.Timestamp(tgl_test[0]))
        fig.add_vrect(
            x0=x_split_train_test, x1=x_split_train_test,
            line_width=1.5, line_color="gray", line_dash="dot",
            annotation_text="Train | Test",
            annotation_position="top left",
            annotation_font_color="gray"
        )

    # Shaded area: batas test/forecast
    if len(tanggal_forecast) > 0:
        x_split_test_fc = str(tanggal_forecast[0])
        fig.add_vrect(
            x0=x_split_test_fc, x1=x_split_test_fc,
            line_width=1.5, line_color="orange", line_dash="dot",
            annotation_text="Test | Forecast",
            annotation_position="top left",
            annotation_font_color="darkorange"
        )

    fig.update_layout(
        title=f"Prediksi {model_dipilih} — {nama_dataset}",
        xaxis_title="Tanggal",
        yaxis_title="Nilai",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        hovermode="x unified",
        height=480
    )
    st.plotly_chart(fig, use_container_width=True)

    # Tabel forecast
    st.subheader("📋 Tabel Forecast ke Depan")
    df_forecast_display = df_forecast.copy()
    df_forecast_display.columns = ["Periode", "Nilai Prediksi"]
    df_forecast_display["Periode"] = df_forecast_display["Periode"].dt.strftime("%Y-%m-%d")
    df_forecast_display["Nilai Prediksi"] = df_forecast_display["Nilai Prediksi"].map("{:.2f}".format)
    st.dataframe(df_forecast_display, use_container_width=True)

# ── Tab 2: Evaluasi Model ─────────────────────────────────────
with tab2:
    mae, rmse, r2 = hitung_metrik(y_test, y_pred_test)
    mae_b, rmse_b, r2_b = hitung_baseline_naive(y_test, y_train[-1])

    col1, col2, col3 = st.columns(3)
    col1.metric(
        "MAE",
        f"{mae:.2f}",
        delta=f"{mae_b - mae:.2f} vs Naive",
        delta_color="inverse"
    )
    col2.metric(
        "RMSE",
        f"{rmse:.2f}",
        delta=f"{rmse_b - rmse:.2f} vs Naive",
        delta_color="inverse"
    )
    col3.metric(
        "R²",
        f"{r2:.4f}",
        delta=f"{r2 - r2_b:.4f} vs Naive",
        delta_color="normal"
    )

    st.caption("Evaluasi dihitung pada test set (20% terakhir data historis)")

    # Scatter: Aktual vs Prediksi
    fig_scatter = go.Figure()

    fig_scatter.add_trace(go.Scatter(
        x=y_test,
        y=y_pred_test,
        mode="markers",
        name="Aktual vs Prediksi",
        marker=dict(color="royalblue", size=8, opacity=0.7),
        hovertemplate="Aktual: %{x:.2f}<br>Prediksi: %{y:.2f}<extra></extra>"
    ))

    # Garis diagonal perfect prediction
    min_val = float(min(y_test.min(), y_pred_test.min()))
    max_val = float(max(y_test.max(), y_pred_test.max()))
    fig_scatter.add_trace(go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode="lines",
        name="Perfect Prediction",
        line=dict(color="gray", dash="dash", width=1.5)
    ))

    fig_scatter.update_layout(
        title="Aktual vs Prediksi (Test Set)",
        xaxis_title="Nilai Aktual",
        yaxis_title="Nilai Prediksi",
        height=400
    )
    st.plotly_chart(fig_scatter, use_container_width=True)

# ── Tab 3: Data Mentah ────────────────────────────────────────
with tab3:
    st.subheader("Data Lengkap")
    df_display = df_data.copy()
    df_display["tanggal"] = pd.to_datetime(df_display["tanggal"]).dt.strftime("%Y-%m-%d")
    df_display["nilai"] = df_display["nilai"].map("{:.2f}".format)
    st.dataframe(df_display, use_container_width=True)

    csv_bytes = df_data.to_csv(index=False).encode("utf-8")
    st.download_button(
        label="⬇️ Download CSV",
        data=csv_bytes,
        file_name=f"data_{nama_dataset.replace(' ', '_')}.csv",
        mime="text/csv"
    )

---
## 👥 CELL 4 — Tulis File `app_segmentasi.py`

---
## 🧠 CELL 5 — Tulis File `app_cnn.py` (Deep Learning - Sesi 4)

In [ ]:
%%writefile app_cnn.py
# ============================================================
# APP 3: CNN IMAGE CLASSIFICATION DEMO — SESI 4
# Deep Learning dengan PyTorch
# ============================================================

import streamlit as st
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ── Konfigurasi halaman ──────────────────────────────────────
st.set_page_config(
    page_title="🧠 CNN Image Classification Demo",
    page_icon="🧠",
    layout="wide"
)

st.title("🧠 CNN Image Classification")
st.markdown(
    "Demo interaktif klasifikasi gambar menggunakan **Convolutional Neural Network (CNN)** "
    "dengan PyTorch. Latih model, coba prediksi gambar, dan bandingkan dengan ML klasik."
)

# ── Konstanta ────────────────────────────────────────────────
CIFAR10_CLASSES = [
    "✈️ Pesawat", "🚗 Mobil", "🐦 Burung", "🐱 Kucing", "🦌 Rusa",
    "🐶 Anjing", "🐸 Katak", "🐴 Kuda", "🚢 Kapal", "🚛 Truk"
]

# ── Definisi arsitektur CNN ──────────────────────────────────
class SimpleCNN(nn.Module):
    """CNN sederhana 2 blok konvolusi untuk CIFAR-10."""
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.blok1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.1)
        )
        self.blok2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.1)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.blok1(x)
        x = self.blok2(x)
        x = self.classifier(x)
        return x


# ── Definisi MLP pembanding ──────────────────────────────────
class SimpleMLP(nn.Module):
    """MLP dasar tanpa konvolusi — sebagai pembanding CNN."""
    def __init__(self, num_classes=10):
        super(SimpleMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 32 * 32, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)


# ── Fungsi: load CIFAR-10 subset ────────────────────────────
@st.cache_data(show_spinner=False)
def load_cifar10_subset(n_train=1000, n_test=500):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010))
    ])
    train_full = torchvision.datasets.CIFAR10(
        root="./data", train=True, download=True, transform=transform
    )
    test_full = torchvision.datasets.CIFAR10(
        root="./data", train=False, download=True, transform=transform
    )
    train_data   = torch.stack([train_full[i][0] for i in range(n_train)])
    train_labels = torch.tensor([train_full[i][1] for i in range(n_train)])
    test_data    = torch.stack([test_full[i][0] for i in range(n_test)])
    test_labels  = torch.tensor([test_full[i][1] for i in range(n_test)])
    return train_data, train_labels, test_data, test_labels


# ── Fungsi: training satu epoch ─────────────────────────────
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_benar = 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss   += loss.item() * len(y_batch)
        total_benar  += (output.argmax(1) == y_batch).sum().item()
    n = len(loader.dataset)
    return total_loss / n, total_benar / n


# ── Fungsi: evaluasi ─────────────────────────────────────────
def evaluasi(model, loader, criterion, device):
    model.eval()
    total_loss, total_benar = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            output = model(X_batch)
            loss = criterion(output, y_batch)
            total_loss  += loss.item() * len(y_batch)
            total_benar += (output.argmax(1) == y_batch).sum().item()
    n = len(loader.dataset)
    return total_loss / n, total_benar / n


# ── Fungsi: prediksi satu gambar ────────────────────────────
def prediksi_gambar(model, img_tensor, device):
    model.eval()
    with torch.no_grad():
        output = model(img_tensor.unsqueeze(0).to(device))
        probs = torch.softmax(output, dim=1).squeeze().cpu().numpy()
    return probs


# ── Fungsi: denormalisasi untuk tampilan ────────────────────
def denormalisasi(tensor):
    mean = np.array([0.4914, 0.4822, 0.4465])
    std  = np.array([0.2023, 0.1994, 0.2010])
    img  = tensor.permute(1, 2, 0).numpy() * std + mean
    return np.clip(img, 0, 1)


# ── Sidebar ──────────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Pengaturan Training")

    st.subheader("📦 Data")
    n_train = st.select_slider(
        "Jumlah Data Latih",
        options=[500, 1000, 2000, 5000], value=1000
    )
    n_test = st.select_slider(
        "Jumlah Data Uji",
        options=[200, 500, 1000], value=500
    )

    st.divider()
    st.subheader("🏗️ Arsitektur")
    pilih_model = st.selectbox(
        "Pilih Model",
        ["CNN (SimpleCNN)", "MLP (Tanpa Konvolusi)"],
        help="Bandingkan CNN vs MLP untuk melihat perbedaan kemampuan menangkap pola spasial"
    )

    st.divider()
    st.subheader("⚡ Hyperparameter")
    n_epoch        = st.slider("Jumlah Epoch", 1, 20, 5)
    batch_size     = st.selectbox("Batch Size", [32, 64, 128], index=1)
    learning_rate  = st.select_slider(
        "Learning Rate",
        options=[0.0001, 0.001, 0.01, 0.1], value=0.001
    )
    optimizer_nama = st.selectbox("Optimizer", ["Adam", "SGD"])

    st.divider()
    with st.expander("ℹ️ Panduan Hyperparameter"):
        st.markdown("""
**Epoch** — berapa kali model melihat seluruh data latih.
Lebih banyak epoch → potensi lebih akurat, tapi risiko overfit.

**Batch Size** — jumlah sampel per update gradien.
Kecil = update lebih sering (lambat tapi stabil).
Besar = update lebih jarang (cepat tapi bisa kurang presisi).

**Learning Rate** — seberapa besar langkah update bobot.
Terlalu besar → tidak konvergen. Terlalu kecil → lambat.

**Optimizer:**
- **Adam** — adaptif, cocok untuk pemula, lebih cepat konvergen.
- **SGD** — klasik, perlu tuning lebih, tapi bisa lebih general.
        """)

    tombol_latih = st.button("🚀 Mulai Training", type="primary", use_container_width=True)


# ── Device & load data ────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with st.spinner("⏳ Mengunduh dataset CIFAR-10 (hanya sekali)..."):
    train_data, train_labels, test_data, test_labels = load_cifar10_subset(n_train, n_test)

train_loader = DataLoader(TensorDataset(train_data, train_labels), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(test_data,  test_labels),  batch_size=batch_size, shuffle=False)

st.info(
    f"💻 Device: **{str(device).upper()}** | "
    f"Data latih: **{n_train}** gambar | "
    f"Data uji: **{n_test}** gambar | "
    f"Dataset: **CIFAR-10** (10 kelas objek)"
)

# ── Inisialisasi session state ────────────────────────────────
for key in ["model_cnn", "history", "model_nama"]:
    if key not in st.session_state:
        st.session_state[key] = None


# ── Proses training ───────────────────────────────────────────
if tombol_latih:
    model = SimpleCNN(10).to(device) if "CNN" in pilih_model else SimpleMLP(10).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = (optim.Adam(model.parameters(), lr=learning_rate)
                 if optimizer_nama == "Adam"
                 else optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9))

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    progress_bar      = st.progress(0, text="Memulai training...")
    status_text       = st.empty()
    chart_placeholder = st.empty()

    for epoch in range(1, n_epoch + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        vl_loss, vl_acc = evaluasi(model, test_loader, criterion, device)
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(vl_loss)
        history["val_acc"].append(vl_acc)

        progress_bar.progress(epoch / n_epoch, text=f"Epoch {epoch}/{n_epoch} — Val Acc: {vl_acc:.2%}")
        status_text.markdown(
            f"**Epoch {epoch}/{n_epoch}** | "
            f"Train Loss: `{tr_loss:.4f}` | Train Acc: `{tr_acc:.2%}` | "
            f"Val Loss: `{vl_loss:.4f}` | Val Acc: `{vl_acc:.2%}`"
        )
        ep_range = list(range(1, epoch + 1))
        fig_live = go.Figure()
        fig_live.add_trace(go.Scatter(x=ep_range, y=history["train_acc"],
            mode="lines+markers", name="Train Acc", line=dict(color="royalblue")))
        fig_live.add_trace(go.Scatter(x=ep_range, y=history["val_acc"],
            mode="lines+markers", name="Val Acc", line=dict(color="seagreen", dash="dash")))
        fig_live.update_layout(title="Akurasi Training (Live)", xaxis_title="Epoch",
            yaxis_title="Accuracy", yaxis=dict(range=[0, 1]), height=300,
            legend=dict(orientation="h", y=1.1))
        chart_placeholder.plotly_chart(fig_live, use_container_width=True)

    st.session_state["model_cnn"]  = model
    st.session_state["history"]    = history
    st.session_state["model_nama"] = pilih_model
    st.success(f"✅ Training selesai! Val Accuracy akhir: **{history['val_acc'][-1]:.2%}**")


# ── TABS ─────────────────────────────────────────────────────
tab1, tab2, tab3, tab4 = st.tabs([
    "📈 Hasil Training",
    "🔍 Prediksi Gambar",
    "🏗️ Arsitektur CNN",
    "⚖️ CNN vs ML Klasik"
])


# ────────────────────────────────────────────────────────────
# Tab 1: Hasil Training
# ────────────────────────────────────────────────────────────
with tab1:
    if st.session_state["history"] is None:
        st.info("⬅️ Atur parameter di sidebar lalu klik **Mulai Training** untuk melihat hasil.")
    else:
        history      = st.session_state["history"]
        ep_range     = list(range(1, len(history["train_acc"]) + 1))

        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Train Accuracy", f"{history['train_acc'][-1]:.2%}")
        c2.metric("Val Accuracy",   f"{history['val_acc'][-1]:.2%}")
        c3.metric("Train Loss",     f"{history['train_loss'][-1]:.4f}")
        c4.metric("Val Loss",       f"{history['val_loss'][-1]:.4f}")

        fig_acc = go.Figure()
        fig_acc.add_trace(go.Scatter(x=ep_range, y=history["train_acc"],
            mode="lines+markers", name="Train Accuracy", line=dict(color="royalblue", width=2)))
        fig_acc.add_trace(go.Scatter(x=ep_range, y=history["val_acc"],
            mode="lines+markers", name="Val Accuracy", line=dict(color="seagreen", width=2, dash="dash")))
        fig_acc.update_layout(
            title=f"Akurasi per Epoch — {st.session_state['model_nama']}",
            xaxis_title="Epoch", yaxis_title="Accuracy",
            xaxis=dict(tickmode="linear", dtick=1),
            yaxis=dict(range=[0, 1]),
            legend=dict(orientation="h", y=1.1), height=380
        )
        st.plotly_chart(fig_acc, use_container_width=True)

        fig_loss = go.Figure()
        fig_loss.add_trace(go.Scatter(x=ep_range, y=history["train_loss"],
            mode="lines+markers", name="Train Loss", line=dict(color="crimson", width=2)))
        fig_loss.add_trace(go.Scatter(x=ep_range, y=history["val_loss"],
            mode="lines+markers", name="Val Loss", line=dict(color="darkorange", width=2, dash="dash")))
        fig_loss.update_layout(
            title="Loss per Epoch", xaxis_title="Epoch", yaxis_title="Loss",
            xaxis=dict(tickmode="linear", dtick=1),
            legend=dict(orientation="h", y=1.1), height=350
        )
        st.plotly_chart(fig_loss, use_container_width=True)

        gap = history["train_acc"][-1] - history["val_acc"][-1]
        if gap > 0.15:
            st.warning(
                f"⚠️ **Potensi Overfitting** — Gap train vs val: `{gap:.2%}`. "
                "Coba kurangi epoch, tambah dropout, atau perbanyak data."
            )
        elif history["val_acc"][-1] < 0.3:
            st.warning(
                "⚠️ **Akurasi rendah (Underfitting)** — Coba tambah epoch, "
                "naikkan learning rate, atau gunakan lebih banyak data."
            )
        else:
            st.success("✅ Training berjalan normal — tidak ada tanda overfitting yang signifikan.")


# ────────────────────────────────────────────────────────────
# Tab 2: Prediksi Gambar
# ────────────────────────────────────────────────────────────
with tab2:
    if st.session_state["model_cnn"] is None:
        st.info("⬅️ Latih model terlebih dahulu sebelum mencoba prediksi.")
    else:
        model = st.session_state["model_cnn"]

        st.subheader("Pilih Sumber Gambar")

        # ── Dua opsi input yang jelas ────────────────────────
        opsi = st.radio(
            "",
            ["🎲 Gunakan Sampel dari Dataset CIFAR-10",
             "📁 Upload Gambar Sendiri"],
            horizontal=True,
            label_visibility="collapsed"
        )

        img_tensor = None
        label_asli = None

        # ── Opsi 1: Sampel CIFAR-10 ──────────────────────────
        if opsi == "🎲 Gunakan Sampel dari Dataset CIFAR-10":
            st.markdown(
                "Dataset CIFAR-10 berisi 10 kelas: "
                "✈️ Pesawat, 🚗 Mobil, 🐦 Burung, 🐱 Kucing, 🦌 Rusa, "
                "🐶 Anjing, 🐸 Katak, 🐴 Kuda, 🚢 Kapal, 🚛 Truk"
            )

            col_filter, col_acak = st.columns([2, 1])
            with col_filter:
                filter_kelas = st.selectbox(
                    "Filter berdasarkan kelas (opsional)",
                    ["Semua Kelas"] + CIFAR10_CLASSES
                )
            with col_acak:
                st.markdown("<br>", unsafe_allow_html=True)
                acak = st.button("🎲 Acak Gambar", use_container_width=True)

            # Tentukan indeks valid berdasarkan filter kelas
            if filter_kelas == "Semua Kelas":
                indeks_valid = list(range(len(test_data)))
            else:
                kelas_idx = CIFAR10_CLASSES.index(filter_kelas)
                indeks_valid = [i for i in range(len(test_labels))
                                if test_labels[i].item() == kelas_idx]

            # Simpan indeks terpilih di session state
            if acak or "idx_sampel" not in st.session_state:
                if indeks_valid:
                    st.session_state["idx_sampel"] = int(
                        np.random.choice(indeks_valid)
                    )

            # Pastikan indeks masih valid setelah filter berubah
            if st.session_state.get("idx_sampel") not in indeks_valid:
                st.session_state["idx_sampel"] = indeks_valid[0] if indeks_valid else 0

            idx_terpilih = st.session_state["idx_sampel"]

            # Slider untuk navigasi manual
            idx_slider = st.slider(
                "Atau pilih indeks gambar secara manual",
                min_value=0,
                max_value=len(indeks_valid) - 1,
                value=indeks_valid.index(idx_terpilih) if idx_terpilih in indeks_valid else 0,
                help="Geser untuk melihat gambar lain"
            )
            idx_terpilih = indeks_valid[idx_slider]
            st.session_state["idx_sampel"] = idx_terpilih

            img_tensor  = test_data[idx_terpilih]
            label_asli  = test_labels[idx_terpilih].item()

            # Tampilkan gambar langsung (diperbesar)
            img_display = denormalisasi(img_tensor)
            col_img, col_meta = st.columns([1, 3])
            with col_img:
                # Perbesar gambar 32x32 → tampil 160px
                pil_besar = Image.fromarray((img_display * 255).astype(np.uint8)).resize(
                    (160, 160), Image.NEAREST
                )
                st.image(pil_besar, caption="Gambar asli (32×32, diperbesar)")
            with col_meta:
                st.markdown(f"**Indeks:** `{idx_terpilih}`")
                st.markdown(f"**Kelas asli:** {CIFAR10_CLASSES[label_asli]}")
                st.markdown(f"**Ukuran asli:** 32 × 32 piksel × 3 channel (RGB)")

        # ── Opsi 2: Upload Gambar Sendiri ────────────────────
        else:
            st.markdown(
                "Upload gambar dari perangkat Anda. "
                "Model akan otomatis meresize ke **32×32 piksel** sesuai format CIFAR-10."
            )
            st.caption(
                "💡 Tips: Untuk hasil terbaik, gunakan gambar yang termasuk salah satu "
                "dari 10 kelas CIFAR-10 dengan objek yang jelas dan latar tidak terlalu ramai."
            )

            file_up = st.file_uploader(
                "Pilih file gambar (JPG / PNG)",
                type=["jpg", "jpeg", "png"],
                help="Gambar akan otomatis diresize ke 32×32 piksel"
            )

            if file_up is not None:
                pil_ori  = Image.open(file_up).convert("RGB")
                pil_kecil = pil_ori.resize((32, 32))
                # Tampilkan dua versi: asli dan 32x32
                col_ori, col_kecil, col_info = st.columns([2, 1, 2])
                with col_ori:
                    st.image(pil_ori, caption=f"Gambar asli ({pil_ori.width}×{pil_ori.height}px)", use_container_width=True)
                with col_kecil:
                    pil_besar = pil_kecil.resize((160, 160), Image.NEAREST)
                    st.image(pil_besar, caption="Setelah resize (32×32)")
                with col_info:
                    st.markdown("**Proses yang terjadi:**")
                    st.markdown(f"1. Gambar asli: `{pil_ori.width}×{pil_ori.height}` px")
                    st.markdown("2. Resize → `32×32` px")
                    st.markdown("3. Normalisasi nilai piksel")
                    st.markdown("4. Masuk ke CNN → probabilitas 10 kelas")

                transform_infer = transforms.Compose([
                    transforms.ToTensor(),
                    transforms.Normalize((0.4914, 0.4822, 0.4465),
                                         (0.2023, 0.1994, 0.2010))
                ])
                img_tensor = transform_infer(pil_kecil)
                st.warning(
                    "⚠️ Model dilatih khusus pada dataset CIFAR-10. "
                    "Akurasi pada gambar di luar distribusi dataset bisa lebih rendah — "
                    "ini adalah fenomena normal yang disebut **distribusi shift**."
                )
            else:
                st.info("⬆️ Belum ada gambar yang diupload.")

        # ── Hasil prediksi (berlaku untuk kedua opsi) ────────
        if img_tensor is not None:
            st.divider()
            st.subheader("🎯 Hasil Prediksi")

            probs    = prediksi_gambar(model, img_tensor, device)
            pred_idx = int(np.argmax(probs))

            col_hasil, col_chart = st.columns([1, 2])
            with col_hasil:
                st.metric("Prediksi Model", CIFAR10_CLASSES[pred_idx])
                st.metric("Confidence", f"{probs[pred_idx]:.2%}")
                if label_asli is not None:
                    if pred_idx == label_asli:
                        st.success("✅ Prediksi **BENAR**")
                    else:
                        st.error(f"❌ Prediksi **SALAH**\n\nKelas asli: {CIFAR10_CLASSES[label_asli]}")

            with col_chart:
                warna_bar = [
                    "#4CAF50" if i == pred_idx else
                    ("#2196F3" if label_asli is not None and i == label_asli else "#E0E0E0")
                    for i in range(10)
                ]
                fig_prob = go.Figure(go.Bar(
                    x=[CIFAR10_CLASSES[i] for i in range(10)],
                    y=probs,
                    marker_color=warna_bar,
                    text=[f"{p:.1%}" for p in probs],
                    textposition="outside",
                    hovertemplate="%{x}<br>Probabilitas: %{y:.2%}<extra></extra>"
                ))
                fig_prob.update_layout(
                    title="Distribusi Probabilitas per Kelas",
                    xaxis_title="Kelas", yaxis_title="Probabilitas",
                    yaxis=dict(range=[0, 1.1]),
                    height=400
                )
                st.plotly_chart(fig_prob, use_container_width=True)
                if label_asli is not None:
                    st.caption("🟢 Hijau = prediksi model | 🔵 Biru = kelas asli")

        # ── Galeri contoh per kelas ──────────────────────────
        with st.expander("🖼️ Galeri Contoh Gambar CIFAR-10 (1 sampel per kelas)"):
            cols = st.columns(10)
            for cls_idx in range(10):
                for i in range(len(test_labels)):
                    if test_labels[i].item() == cls_idx:
                        img_ex   = denormalisasi(test_data[i])
                        pil_ex   = Image.fromarray((img_ex * 255).astype(np.uint8)).resize(
                            (64, 64), Image.NEAREST
                        )
                        cols[cls_idx].image(pil_ex, caption=CIFAR10_CLASSES[cls_idx],
                                            use_container_width=True)
                        break


# ────────────────────────────────────────────────────────────
# Tab 3: Arsitektur CNN
# ────────────────────────────────────────────────────────────
with tab3:
    st.subheader("🏗️ Diagram Arsitektur SimpleCNN")

    arsitektur = [
        {"Layer": "Input",              "Tipe": "—",                      "Detail": "3 × 32 × 32 (RGB)",                    "Output Shape": "3 × 32 × 32"},
        {"Layer": "Conv2d-1",           "Tipe": "Konvolusi",               "Detail": "32 filter, kernel 3×3, padding 1",     "Output Shape": "32 × 32 × 32"},
        {"Layer": "BatchNorm + ReLU",   "Tipe": "Normalisasi + Aktivasi",  "Detail": "Normalisasi per batch, non-linear",    "Output Shape": "32 × 32 × 32"},
        {"Layer": "Conv2d-2",           "Tipe": "Konvolusi",               "Detail": "32 filter, kernel 3×3, padding 1",     "Output Shape": "32 × 32 × 32"},
        {"Layer": "MaxPool2d",          "Tipe": "Pooling",                 "Detail": "Kernel 2×2, stride 2",                 "Output Shape": "32 × 16 × 16"},
        {"Layer": "Dropout2d (p=0.1)",  "Tipe": "Regularisasi",            "Detail": "10% channel di-nol-kan secara acak",   "Output Shape": "32 × 16 × 16"},
        {"Layer": "Conv2d-3",           "Tipe": "Konvolusi",               "Detail": "64 filter, kernel 3×3, padding 1",     "Output Shape": "64 × 16 × 16"},
        {"Layer": "BatchNorm + ReLU",   "Tipe": "Normalisasi + Aktivasi",  "Detail": "Normalisasi per batch, non-linear",    "Output Shape": "64 × 16 × 16"},
        {"Layer": "Conv2d-4",           "Tipe": "Konvolusi",               "Detail": "64 filter, kernel 3×3, padding 1",     "Output Shape": "64 × 16 × 16"},
        {"Layer": "MaxPool2d",          "Tipe": "Pooling",                 "Detail": "Kernel 2×2, stride 2",                 "Output Shape": "64 × 8 × 8"},
        {"Layer": "Flatten",            "Tipe": "Reshape",                 "Detail": "64 × 8 × 8 → vektor 1D",              "Output Shape": "4.096"},
        {"Layer": "Linear-1",           "Tipe": "Fully Connected",         "Detail": "4.096 → 256 neuron",                  "Output Shape": "256"},
        {"Layer": "Dropout (p=0.4)",    "Tipe": "Regularisasi",            "Detail": "40% neuron di-nol-kan secara acak",    "Output Shape": "256"},
        {"Layer": "Linear-2 (Output)",  "Tipe": "Fully Connected",         "Detail": "256 → 10 neuron (1 per kelas)",        "Output Shape": "10"},
        {"Layer": "Softmax",            "Tipe": "Aktivasi Output",         "Detail": "Konversi logit → probabilitas",        "Output Shape": "10 probabilitas"},
    ]
    st.dataframe(pd.DataFrame(arsitektur), use_container_width=True, hide_index=True)

    st.subheader("📊 Alur Dimensi Data per Layer")
    layer_names = ["Input\n3×32×32", "Blok1 Conv\n32×32×32", "Blok1 Pool\n32×16×16",
                   "Blok2 Conv\n64×16×16", "Blok2 Pool\n64×8×8",
                   "Flatten\n4096", "FC-1\n256", "Output\n10"]
    layer_sizes = [3*32*32, 32*32*32, 32*16*16, 64*16*16, 64*8*8, 4096, 256, 10]
    warna_layer = ["#2196F3","#4CAF50","#4CAF50","#FF9800","#FF9800","#9C27B0","#E91E63","#F44336"]

    fig_dim = go.Figure(go.Bar(
        x=layer_names, y=layer_sizes, marker_color=warna_layer,
        hovertemplate="%{x}<br>Jumlah nilai: %{y:,}<extra></extra>"
    ))
    fig_dim.update_layout(
        title="Jumlah Nilai per Layer (skala log)",
        xaxis_title="Layer", yaxis_title="Jumlah Nilai",
        yaxis_type="log", height=380
    )
    st.plotly_chart(fig_dim, use_container_width=True)

    st.subheader("📚 Konsep Kunci CNN")
    ca, cb, cc = st.columns(3)
    with ca:
        st.markdown("""
**🔲 Konvolusi (Conv2d)**
Filter kecil bergeser di atas gambar untuk mendeteksi pola lokal.
- Layer awal → tepi, warna
- Layer dalam → bentuk, tekstur kompleks
- **Parameter sharing**: filter yang sama dipakai di semua posisi → jauh lebih efisien dari MLP
        """)
    with cb:
        st.markdown("""
**📉 Max Pooling**
Ambil nilai maksimum dari tiap region kecil → reduksi dimensi.
- Kurangi parameter secara drastis
- Buat representasi lebih ringkas
- **Translasi invariance**: objek yang sedikit bergeser tetap terdeteksi
        """)
    with cc:
        st.markdown("""
**🛡️ Batch Norm + Dropout**
- **BatchNorm**: normalisasi aktivasi per batch → training lebih stabil
- **Dropout**: matikan neuron secara acak → cegah overfit, paksa model belajar lebih robust
        """)

    if st.session_state["model_cnn"] is not None:
        m = st.session_state["model_cnn"]
        total = sum(p.numel() for p in m.parameters())
        train = sum(p.numel() for p in m.parameters() if p.requires_grad)
        st.info(f"🔢 **Parameter {st.session_state['model_nama']}:** Total = `{total:,}` | Trainable = `{train:,}`")
    else:
        total = sum(p.numel() for p in SimpleCNN().parameters())
        st.info(f"🔢 **Parameter SimpleCNN (default):** Total = `{total:,}`")


# ────────────────────────────────────────────────────────────
# Tab 4: CNN vs ML Klasik
# ────────────────────────────────────────────────────────────
with tab4:
    st.subheader("⚖️ Perbandingan CNN vs ML Klasik pada CIFAR-10")
    st.markdown(
        "Benchmark ini melatih **SVM** dan **Random Forest** pada fitur piksel mentah "
        "(gambar di-*flatten* jadi vektor 1D), lalu dibandingkan dengan CNN yang sudah Anda latih."
    )

    n_compare    = st.slider("Jumlah data untuk benchmark ML Klasik", 200, 1000, 500,
                             help="ML klasik lebih lambat — gunakan data lebih sedikit untuk perbandingan cepat")
    tombol_bench = st.button("⚡ Jalankan Benchmark", type="secondary")

    if tombol_bench:
        X_tr = train_data[:n_compare].reshape(n_compare, -1).numpy()
        y_tr = train_labels[:n_compare].numpy()
        X_te = test_data[:n_compare].reshape(n_compare, -1).numpy()
        y_te = test_labels[:n_compare].numpy()

        hasil = {}
        with st.spinner("Training SVM (RBF)..."):
            svm = SVC(kernel="rbf", C=1.0, random_state=42)
            svm.fit(X_tr, y_tr)
            hasil["SVM (RBF)"] = accuracy_score(y_te, svm.predict(X_te))

        with st.spinner("Training Random Forest..."):
            rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
            rf.fit(X_tr, y_tr)
            hasil["Random Forest"] = accuracy_score(y_te, rf.predict(X_te))

        if st.session_state["model_cnn"] is not None:
            crit  = nn.CrossEntropyLoss()
            ldr   = DataLoader(TensorDataset(test_data[:n_compare], test_labels[:n_compare]), batch_size=64)
            _, acc = evaluasi(st.session_state["model_cnn"], ldr, crit, device)
            hasil[st.session_state["model_nama"]] = acc

        st.session_state["benchmark"] = hasil

    if st.session_state.get("benchmark"):
        hasil       = st.session_state["benchmark"]
        nama_list   = list(hasil.keys())
        acc_list    = list(hasil.values())

        warna = ["#4CAF50" if "CNN" in n else ("#FF9800" if "MLP" in n else "#2196F3")
                 for n in nama_list]

        fig_b = go.Figure(go.Bar(
            x=nama_list, y=acc_list, marker_color=warna,
            text=[f"{v:.2%}" for v in acc_list], textposition="outside",
            hovertemplate="%{x}<br>Akurasi: %{y:.2%}<extra></extra>"
        ))
        fig_b.update_layout(
            title="Perbandingan Akurasi: CNN vs ML Klasik (Test Set)",
            xaxis_title="Model", yaxis_title="Akurasi",
            yaxis=dict(range=[0, 1.1]), height=420
        )
        st.plotly_chart(fig_b, use_container_width=True)

        df_b = pd.DataFrame({
            "Model":  nama_list,
            "Akurasi": [f"{v:.2%}" for v in acc_list],
            "Tipe":   ["Deep Learning" if ("CNN" in n or "MLP" in n) else "ML Klasik"
                       for n in nama_list]
        })
        st.dataframe(df_b, use_container_width=True, hide_index=True)

        st.subheader("💡 Insight")
        best     = max(hasil, key=hasil.get)
        ml_vals  = {k: v for k, v in hasil.items() if "CNN" not in k and "MLP" not in k}
        if ml_vals:
            best_ml, best_ml_acc = max(ml_vals.items(), key=lambda x: x[1])
            if "CNN" in best:
                selisih = hasil[best] - best_ml_acc
                st.success(
                    f"🏆 **CNN unggul {selisih:.2%}** dibanding ML klasik terbaik "
                    f"({best_ml}: {best_ml_acc:.2%}). "
                    "CNN mampu menangkap pola spasial gambar yang tidak bisa dilakukan "
                    "SVM/RF yang hanya melihat piksel mentah."
                )
            else:
                st.info(
                    f"📊 Model terbaik saat ini: **{best}** ({hasil[best]:.2%}). "
                    "CNN mungkin perlu lebih banyak epoch atau data. Coba latih lebih lama!"
                )

        st.markdown("""
**Mengapa CNN lebih unggul untuk data gambar?**

| Aspek | ML Klasik (SVM / RF) | CNN |
|-------|----------------------|-----|
| Format input | Vektor piksel datar (flat) | Tensor spasial H × W × C |
| Cara ekstraksi fitur | Manual / piksel mentah | Otomatis dipelajari (feature learning) |
| Pola spasial | ❌ Tidak bisa menangkap | ✅ Konvolusi menangkap pola lokal |
| Translasi invariance | ❌ Tidak ada | ✅ Max Pooling memberi invariance |
| Skalabilitas | ❌ Lambat di gambar besar | ✅ Parameter sharing sangat efisien |
        """)

In [ ]:
%%writefile app_segmentasi.py
# ============================================================
# APP 2: CUSTOMER SEGMENTATION DEMO
# Unsupervised Learning — KMeans Clustering
# ============================================================

import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# ── Konfigurasi halaman ──────────────────────────────────────
st.set_page_config(
    page_title="👥 Customer Segmentation Demo",
    page_icon="👥",
    layout="wide"
)

st.title("👥 Customer Segmentation")
st.markdown(
    "Demo interaktif segmentasi pelanggan menggunakan "
    "Unsupervised Learning (KMeans Clustering). "
    "Geser slider K dan eksplorasi bagaimana cluster terbentuk."
)

# ── Fungsi: buat data demo ───────────────────────────────────
def buat_data_demo():
    """Generate 200 pelanggan sintetis dengan 3 cluster, seed=42."""
    rng = np.random.RandomState(42)

    # Cluster A: Pelanggan Pasif
    n_a = 60
    usia_a = rng.uniform(20, 30, n_a)
    pengeluaran_a = rng.uniform(200, 400, n_a)
    frekuensi_a = rng.uniform(1, 3, n_a)

    # Cluster B: Pelanggan Potensial
    n_b = 80
    usia_b = rng.uniform(30, 50, n_b)
    pengeluaran_b = rng.uniform(500, 900, n_b)
    frekuensi_b = rng.uniform(4, 7, n_b)

    # Cluster C: Pelanggan Premium
    n_c = 60
    usia_c = rng.uniform(35, 60, n_c)
    pengeluaran_c = rng.uniform(1000, 2000, n_c)
    frekuensi_c = rng.uniform(8, 15, n_c)

    usia = np.concatenate([usia_a, usia_b, usia_c])
    pengeluaran = np.concatenate([pengeluaran_a, pengeluaran_b, pengeluaran_c])
    frekuensi = np.concatenate([frekuensi_a, frekuensi_b, frekuensi_c])

    # Tambah noise kecil agar tidak terlalu perfect
    usia += rng.normal(0, 1, len(usia))
    pengeluaran += rng.normal(0, 20, len(pengeluaran))
    frekuensi += rng.normal(0, 0.3, len(frekuensi))

    df = pd.DataFrame({
        "usia": np.round(usia, 1),
        "pengeluaran_bulanan": np.round(pengeluaran, 1),
        "frekuensi_transaksi": np.round(frekuensi, 1)
    })
    return df

# ── Fungsi: assign label interpretatif ──────────────────────
def assign_label_cluster(labels_array, df_asli, kolom_proxy, k, is_demo):
    """
    Ranking cluster berdasarkan rata-rata kolom_proxy.
    Tertinggi → Premium, Terendah → Pasif, Tengah → Potensial.
    """
    rata_per_cluster = {}
    for c in range(k):
        mask = labels_array == c
        if mask.sum() > 0:
            rata_per_cluster[c] = df_asli.loc[mask, kolom_proxy].mean()
        else:
            rata_per_cluster[c] = 0

    # Urutkan cluster dari pengeluaran tertinggi ke terendah
    sorted_clusters = sorted(rata_per_cluster.items(), key=lambda x: x[1], reverse=True)

    label_map = {}
    n_middle = k - 2  # jumlah cluster "Potensial"

    for rank, (cluster_id, _) in enumerate(sorted_clusters):
        if rank == 0:
            if is_demo:
                label_map[cluster_id] = "💎 Cluster Premium / Premium Customers"
            else:
                label_map[cluster_id] = "🔵 Cluster Tinggi / High"
        elif rank == len(sorted_clusters) - 1:
            if is_demo:
                label_map[cluster_id] = "😴 Cluster Pasif / Passive Customers"
            else:
                label_map[cluster_id] = "🔴 Cluster Rendah / Low"
        else:
            middle_rank = rank - 1  # index di antara Tinggi dan Rendah
            if n_middle == 1:
                if is_demo:
                    label_map[cluster_id] = "🌱 Cluster Potensial / Potential Customers"
                else:
                    label_map[cluster_id] = "🟡 Cluster Menengah / Medium"
            else:
                if is_demo:
                    label_map[cluster_id] = f"🌱 Cluster Potensial {middle_rank + 1}"
                else:
                    label_map[cluster_id] = f"🟡 Cluster Menengah {middle_rank + 1}"

    return label_map

# ── Fungsi: buat narasi cluster dinamis ─────────────────────
def buat_narasi_cluster(label, rata_rata_dict, is_demo):
    """Buat teks deskripsi dan rekomendasi berdasarkan label dan nilai rata-rata."""
    fitur_str = ", ".join([f"**{k}**: {v:.2f}" for k, v in rata_rata_dict.items()])

    if "Premium" in label or "Tinggi" in label:
        emoji = "💎"
        deskripsi = (
            f"Cluster ini memiliki profil tertinggi di antara semua kelompok. "
            f"Rata-rata fitur: {fitur_str}. "
            f"Pelanggan di cluster ini menunjukkan keterlibatan dan nilai transaksi yang sangat tinggi."
        )
        rekomendasi = (
            "💡 **Rekomendasi:** Pertahankan dengan program loyalitas eksklusif, "
            "penawaran VIP, dan layanan personal. Jangan kehilangan segmen ini."
        )
    elif "Pasif" in label or "Rendah" in label:
        emoji = "😴"
        deskripsi = (
            f"Cluster ini memiliki profil terendah di antara semua kelompok. "
            f"Rata-rata fitur: {fitur_str}. "
            f"Pelanggan di cluster ini jarang bertransaksi dan cenderung pasif."
        )
        rekomendasi = (
            "💡 **Rekomendasi:** Aktifkan kembali dengan diskon re-engagement, "
            "notifikasi promosi, atau kampanye win-back."
        )
    else:
        emoji = "🌱"
        deskripsi = (
            f"Cluster ini memiliki profil menengah. "
            f"Rata-rata fitur: {fitur_str}. "
            f"Pelanggan di cluster ini memiliki potensi untuk naik ke tier premium."
        )
        rekomendasi = (
            "💡 **Rekomendasi:** Dorong dengan program upsell dan reward bertahap. "
            "Target konversi ke tier premium melalui penawaran yang tepat sasaran."
        )

    return emoji, deskripsi, rekomendasi

# ── Sidebar ──────────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Pengaturan")

    sumber_data = st.radio(
        "Pilih Sumber Data",
        options=["👥 Data Demo (Pelanggan Sintetis)", "📁 Upload CSV Sendiri"]
    )

    df_data = None
    fitur_cols = None
    is_demo = True

    # ── Opsi 1: Data Demo ────────────────────────────────────
    if sumber_data == "👥 Data Demo (Pelanggan Sintetis)":
        df_data = buat_data_demo()
        fitur_cols = ["usia", "pengeluaran_bulanan", "frekuensi_transaksi"]
        is_demo = True

    # ── Opsi 2: Upload CSV ───────────────────────────────────
    else:
        is_demo = False
        st.markdown(
            "**Format:** Minimal 2 kolom numerik, maksimal 10 kolom. "
            "Kolom non-numerik diabaikan otomatis."
        )
        file_upload = st.file_uploader("Upload file CSV", type=["csv"])

        if file_upload is not None:
            try:
                df_upload = pd.read_csv(file_upload)
                # Ambil hanya kolom numerik
                df_numerik = df_upload.select_dtypes(include=[np.number])
                if len(df_numerik.columns) < 2:
                    st.error("❌ File harus memiliki minimal 2 kolom numerik.")
                else:
                    kolom_numerik = list(df_numerik.columns[:10])
                    st.write("**Preview 5 baris pertama:**")
                    st.dataframe(df_upload.head())
                    fitur_terpilih = st.multiselect(
                        "Pilih kolom untuk clustering (min. 2)",
                        options=kolom_numerik,
                        default=kolom_numerik[:min(3, len(kolom_numerik))]
                    )
                    if len(fitur_terpilih) >= 2:
                        df_data = df_upload[fitur_terpilih].dropna().reset_index(drop=True)
                        fitur_cols = fitur_terpilih
                        st.success(f"✅ Data berhasil diproses: {len(df_data)} baris, {len(fitur_cols)} fitur")
                    else:
                        st.warning("⚠️ Pilih minimal 2 kolom untuk melanjutkan.")
            except Exception as e:
                st.error(f"❌ Gagal membaca file: {e}")

        # Fallback ke demo jika upload belum ada
        if df_data is None:
            df_data = buat_data_demo()
            fitur_cols = ["usia", "pengeluaran_bulanan", "frekuensi_transaksi"]
            is_demo = True

    st.divider()

    # Slider jumlah cluster
    k = st.slider("Jumlah Cluster (K)", min_value=2, max_value=6, value=3)

    # Checkbox interpretasi
    tampilkan_interpretasi = st.checkbox("💡 Tampilkan Interpretasi Cluster", value=False)

# ── Pastikan data tersedia ────────────────────────────────────
if df_data is None or fitur_cols is None:
    st.error("Data tidak tersedia.")
    st.stop()

# ── Proses: Scaling + KMeans ──────────────────────────────────
X = df_data[fitur_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
labels = kmeans.fit_predict(X_scaled)
df_data = df_data.copy()
df_data["cluster_id"] = labels

# ── Kolom proxy untuk interpretasi ───────────────────────────
if is_demo and "pengeluaran_bulanan" in fitur_cols:
    kolom_proxy = "pengeluaran_bulanan"
else:
    kolom_proxy = fitur_cols[0]

# ── Map label cluster ─────────────────────────────────────────
label_map = assign_label_cluster(labels, df_data, kolom_proxy, k, is_demo)

if tampilkan_interpretasi:
    df_data["cluster_label"] = df_data["cluster_id"].map(label_map)
else:
    df_data["cluster_label"] = df_data["cluster_id"].map(
        {c: f"Cluster {c}" for c in range(k)}
    )

# ── PCA 2D untuk visualisasi ──────────────────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df_data["pca_x"] = X_pca[:, 0]
df_data["pca_y"] = X_pca[:, 1]

# Centroid PCA
centroid_scaled = kmeans.cluster_centers_
centroid_pca = pca.transform(centroid_scaled)

# ── Warna cluster (color-blind friendly) ─────────────────────
WARNA_CLUSTER = [
    "#2196F3",  # biru
    "#FF9800",  # oranye
    "#4CAF50",  # hijau
    "#E91E63",  # merah muda
    "#9C27B0",  # ungu
    "#00BCD4",  # cyan
]

# ── TABS ─────────────────────────────────────────────────────
tab1, tab2, tab3 = st.tabs([
    "🗺️ Visualisasi Cluster",
    "📊 Profil Cluster",
    "📉 Elbow Method"
])

# ── Tab 1: Visualisasi Cluster ────────────────────────────────
with tab1:
    fig = go.Figure()

    for idx_c in range(k):
        mask = df_data["cluster_id"] == idx_c
        subset = df_data[mask]
        label_c = subset["cluster_label"].iloc[0] if len(subset) > 0 else f"Cluster {idx_c}"
        warna = WARNA_CLUSTER[idx_c % len(WARNA_CLUSTER)]

        # Teks hover: semua fitur asli
        hover_teks = [
            "<br>".join([f"{col}: {row[col]:.2f}" for col in fitur_cols])
            + f"<br><b>{label_c}</b>"
            for _, row in subset.iterrows()
        ]

        fig.add_trace(go.Scatter(
            x=subset["pca_x"],
            y=subset["pca_y"],
            mode="markers",
            name=label_c,
            marker=dict(color=warna, size=8, opacity=0.75),
            text=hover_teks,
            hovertemplate="%{text}<extra></extra>"
        ))

    # Centroid sebagai bintang
    for idx_c in range(k):
        label_c = label_map.get(idx_c, f"Cluster {idx_c}")
        fig.add_trace(go.Scatter(
            x=[centroid_pca[idx_c, 0]],
            y=[centroid_pca[idx_c, 1]],
            mode="markers",
            name=f"Centroid {idx_c}",
            marker=dict(
                symbol="star",
                size=20,
                color="black",
                line=dict(color="white", width=1.5)
            ),
            showlegend=False,
            hovertemplate=f"Centroid {label_c}<extra></extra>"
        ))

    fig.update_layout(
        title=f"Visualisasi Cluster (K={k}) — PCA 2D",
        xaxis_title="PCA Komponen 1",
        yaxis_title="PCA Komponen 2",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        height=500
    )
    st.plotly_chart(fig, use_container_width=True)
    st.caption(
        "📌 PCA digunakan hanya untuk visualisasi 2D. "
        "Proses clustering dilakukan pada data asli setelah scaling."
    )

# ── Tab 2: Profil Cluster ─────────────────────────────────────
with tab2:
    # Tabel rata-rata per cluster
    df_profil = df_data.groupby("cluster_label")[fitur_cols].mean().round(2)
    st.subheader("Rata-rata Fitur per Cluster")
    st.dataframe(df_profil, use_container_width=True)

    # Grouped bar chart
    df_profil_reset = df_profil.reset_index()
    fig_bar = go.Figure()

    for idx_c, row in df_profil_reset.iterrows():
        label_c = row["cluster_label"]
        warna = WARNA_CLUSTER[idx_c % len(WARNA_CLUSTER)]
        fig_bar.add_trace(go.Bar(
            name=label_c,
            x=fitur_cols,
            y=[row[col] for col in fitur_cols],
            marker_color=warna
        ))

    fig_bar.update_layout(
        title="Perbandingan Rata-rata Fitur per Cluster",
        barmode="group",
        xaxis_title="Fitur",
        yaxis_title="Nilai Rata-rata",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        height=400
    )
    st.plotly_chart(fig_bar, use_container_width=True)

    # Narasi interpretatif jika diaktifkan
    if tampilkan_interpretasi:
        st.subheader("📝 Interpretasi Cluster")
        for idx_c in range(k):
            label_c = label_map.get(idx_c, f"Cluster {idx_c}")
            mask = df_data["cluster_id"] == idx_c
            rata_dict = {
                col: df_data.loc[mask, col].mean()
                for col in fitur_cols
            }
            emoji, deskripsi, rekomendasi = buat_narasi_cluster(label_c, rata_dict, is_demo)

            with st.expander(f"{emoji} {label_c} ({mask.sum()} pelanggan)"):
                st.markdown(deskripsi)
                st.markdown(rekomendasi)

# ── Tab 3: Elbow Method ───────────────────────────────────────
with tab3:
    # Hitung inertia untuk K=1 sampai 10
    k_range = list(range(1, 11))
    inertia_list = []
    for k_test in k_range:
        km_test = KMeans(n_clusters=k_test, n_init=10, random_state=42)
        km_test.fit(X_scaled)
        inertia_list.append(km_test.inertia_)

    fig_elbow = go.Figure()

    # Garis biru: inertia
    fig_elbow.add_trace(go.Scatter(
        x=k_range,
        y=inertia_list,
        mode="lines+markers",
        name="Inertia",
        line=dict(color="royalblue", width=2),
        marker=dict(size=7, color="royalblue"),
        hovertemplate="K=%{x}<br>Inertia=%{y:.2f}<extra></extra>"
    ))

    # Marker merah: K yang sedang dipilih
    idx_k = k - 1  # index di k_range
    fig_elbow.add_trace(go.Scatter(
        x=[k],
        y=[inertia_list[idx_k]],
        mode="markers",
        name=f"K={k} (dipilih)",
        marker=dict(size=15, color="crimson", symbol="circle"),
        hovertemplate=f"K={k}<br>Inertia={inertia_list[idx_k]:.2f}<extra>K dipilih</extra>"
    ))

    fig_elbow.update_layout(
        title="Elbow Method — Mencari K Optimal",
        xaxis_title="Jumlah Cluster (K)",
        yaxis_title="Inertia",
        xaxis=dict(tickmode="linear", dtick=1),
        height=420
    )
    st.plotly_chart(fig_elbow, use_container_width=True)

    st.info(
        "📖 Cara membaca grafik ini: Pilih nilai K di titik 'siku' (elbow), "
        "yaitu titik di mana penurunan inertia mulai melambat secara signifikan. "
        "Menambah K setelah titik ini tidak memberikan manfaat yang sebanding."
    )

---
## 🔮 CELL 5 — Jalankan App 1: Time Series Forecasting

> Tunggu link ngrok muncul, lalu klik untuk membuka di tab baru.

---
## 🧠 CELL 7 — Jalankan App 3: CNN Image Classification (Sesi 4)

> Tunggu link ngrok muncul, lalu klik untuk membuka di tab baru.

In [ ]:
import subprocess, threading, time
from pyngrok import ngrok

def run_app3():
    subprocess.Popen(
        ["streamlit", "run", "app_cnn.py",
         "--server.port", "8503",
         "--server.headless", "true"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

threading.Thread(target=run_app3, daemon=True).start()
time.sleep(5)

tunnel3 = ngrok.connect(8503)
print("=" * 55)
print("🧠 CNN IMAGE CLASSIFICATION APP — SIAP DIGUNAKAN")
print(f"🌐 Buka di browser: {tunnel3.public_url}")
print("=" * 55)

In [ ]:
import subprocess, threading, time
from pyngrok import ngrok

def run_app1():
    subprocess.Popen(
        ["streamlit", "run", "app_timeseries.py",
         "--server.port", "8501",
         "--server.headless", "true"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

threading.Thread(target=run_app1, daemon=True).start()
time.sleep(5)

tunnel1 = ngrok.connect(8501)
print("=" * 55)
print("🔮 TIME SERIES FORECASTING APP — SIAP DIGUNAKAN")
print(f"🌐 Buka di browser: {tunnel1.public_url}")
print("=" * 55)

---
## 👥 CELL 6 — Jalankan App 2: Customer Segmentation

> Tunggu link ngrok muncul, lalu klik untuk membuka di tab baru.

In [ ]:
import subprocess, threading, time
from pyngrok import ngrok

def run_app2():
    subprocess.Popen(
        ["streamlit", "run", "app_segmentasi.py",
         "--server.port", "8502",
         "--server.headless", "true"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

threading.Thread(target=run_app2, daemon=True).start()
time.sleep(5)

tunnel2 = ngrok.connect(8502)
print("=" * 55)
print("👥 CUSTOMER SEGMENTATION APP — SIAP DIGUNAKAN")
print(f"🌐 Buka di browser: {tunnel2.public_url}")
print("=" * 55)

---
## 🛑 CELL 7 — Hentikan Semua Tunnel Ngrok

> **Jangan jalankan cell ini saat Run All!**
>
> Cell ini sengaja dikomentari agar tidak ikut berjalan saat **Runtime → Run all**.
> Jalankan **hanya jika ingin menghentikan** semua tunnel ngrok secara manual.
>
> **Cara pakai:** Hapus tanda `#` pada baris kode di bawah, lalu klik ▶ jalankan cell ini.

In [ ]:
# Hapus tanda # di bawah ini, lalu jalankan cell ini untuk menghentikan semua tunnel

# ngrok.kill()
# print("✅ Semua tunnel ngrok telah dihentikan.")